# Tennis Sensor Analysis: Cloud Parallel Processing

This notebook processes raw EMG/IMU logs from the ESP32 Tennis Sensor system. 
It uses **multiprocessing** to analyze multiple sessions simultaneously, extracting fatigue features and swing biomechanics.

### Prerequisites:
1.  **Google Drive** mounted.
2.  **Python Scripts** (`emg_util.py`, `emg_util_yaml_runner.py`, `capture_loader.py`, `imu_feature.py`) uploaded to a folder in Drive.
3.  **Log Files** (`emg_log_*.csv`, `imu_log_*.csv`) uploaded to a data folder in Drive.

In [ ]:
# --- CELL 1: SETUP & MOUNT DRIVE ---
from google.colab import drive
import sys
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Install dependencies
!pip install pyyaml scipy pandas matplotlib

# =============================================================================
# TODO: UPDATE THIS PATH to where your python scripts (.py) are located in Drive
# =============================================================================
SCRIPT_DIR = '/content/drive/MyDrive/Tennis_Project/scripts'

if not os.path.exists(SCRIPT_DIR):
    print(f"WARNING: Script directory not found: {SCRIPT_DIR}")
    print("Please update SCRIPT_DIR to point to your uploaded python files.")
else:
    sys.path.append(SCRIPT_DIR)
    print(f"Added {SCRIPT_DIR} to Python path.")

In [ ]:
# --- CELL 2: IMPORTS ---
# Import your custom modules. 
# If this fails, check SCRIPT_DIR in Cell 1.

try:
    import emg_util_yaml_runner as runner
    import capture_loader as loader
    import emg_util as core
    print("Successfully imported tennis analysis modules.")
except ImportError as e:
    print(f"ERROR importing modules: {e}")
    print("Make sure emg_util_yaml_runner.py, capture_loader.py, and emg_util.py are in SCRIPT_DIR.")

import pandas as pd
import matplotlib.pyplot as plt
import multiprocessing
from concurrent.futures import ProcessPoolExecutor

In [ ]:
# --- CELL 3: CONFIGURATION ---
# Instead of loading a YAML file, we define the config here for easy cloud experimentation.

# =============================================================================
# TODO: UPDATE DATA PATHS
# =============================================================================
LOG_DIR = '/content/drive/MyDrive/Tennis_Project/raw_logs'       # Where your CSVs are
OUT_DIR = '/content/drive/MyDrive/Tennis_Project/processed_out'  # Where results go

CLOUD_CONFIG = {
    'session': {
        'emg_glob': 'emg_log_*.csv',
        'imu_glob': 'imu_log_*.csv',
        'vitals_path': 'vitals_log.csv',
        'emg_fs_hz': 2000,
        'acc_fsr_default_g': 2,
        'gyro_fsr_default_dps': 250,
        'use_physical_units': True
    },
    'features': {
        'mode': 'both', # 'bout' (fast) or 'both' (deep analysis)
        'zc_thr': 0.01,
        'ssc_thr': 0.01
    },
    'swing_detection': {
        'use_gyro': True,
        'z_thr': 5.0,         # Threshold for swing detection (Z-score)
        'min_consec': 3,
        'min_gap_ms': 300.0,
        'pre_ms': 250.0,
        'post_ms': 450.0
    },
    'baseline': {
        'warmup_skip_min': 5.0,
        'baseline_swings': 20
    },
    'plots': {
        'enable': True,
        'full_duration': True,
        'do_fatigue': True,
        'win_sec': 1.0,
        'step_sec': 0.25
    },
    'diagnostics': {
        'activity_overlay': {
            'enable': True,
            'plot_swings': True
        }
    }
}

print(f"Config loaded. \nInput: {LOG_DIR}\nOutput: {OUT_DIR}")

In [ ]:
# --- CELL 4: DISCOVERY & PARALLEL EXECUTION ---

# 1. Discover Sessions
if not os.path.exists(LOG_DIR):
    print(f"ERROR: Log directory does not exist: {LOG_DIR}")
else:
    print(f"Scanning {LOG_DIR} for sessions...")
    sessions = loader.discover_sessions(
        LOG_DIR,
        emg_glob=CLOUD_CONFIG['session']['emg_glob'],
        imu_glob=CLOUD_CONFIG['session']['imu_glob'],
        vitals_name=CLOUD_CONFIG['session']['vitals_path']
    )
    
    session_ids = sorted(sessions.keys())
    print(f"Found {len(session_ids)} sessions: {session_ids}")

    # 2. Define Worker Wrapper
    def process_wrapper(session_id):
        """Helper to run analysis in a separate process"""
        # Get file paths for this session
        files = sessions[session_id]
        try:
            # Call the refactored function in emg_util_yaml_runner
            # Note: This requires the 'analyze_single_session' function to exist in your runner script
            out_path = runner.analyze_single_session(session_id, files, CLOUD_CONFIG, OUT_DIR)
            return (session_id, "SUCCESS", out_path)
        except Exception as e:
            # Return error string so we don't crash the whole pool
            return (session_id, "FAILED", str(e))

    # 3. Run Parallel Processing
    if session_ids:
        num_workers = multiprocessing.cpu_count()
        print(f"\nStarting parallel processing on {num_workers} CPU cores...\n")
        
        results = []
        with ProcessPoolExecutor(max_workers=num_workers) as executor:
            futures = executor.map(process_wrapper, session_ids)
            
            for res in futures:
                results.append(res)
                status = res[1]
                print(f"[{status}] Session {res[0]}")
                if status == "FAILED":
                    print(f"   Error: {res[2]}")

        print("\n--- Processing Batch Complete ---")

In [ ]:
# --- CELL 5: AGGREGATE RESULTS & CREATE TRAINING SET ---
# This cell combines all individual session results into one Master Dataset
# ready for AI training.

all_swings = []

if 'session_ids' in locals() and session_ids:
    print("Aggregating data from processed sessions...")
    
    for session_id in session_ids:
        # Path to the swing features generated in previous step
        csv_path = os.path.join(OUT_DIR, session_id, 'swing_features.csv')
        
        if os.path.exists(csv_path):
            try:
                df = pd.read_csv(csv_path)
                df['session_id'] = session_id # Tag rows with session ID
                all_swings.append(df)
            except Exception as e:
                print(f"Warning: Could not read CSV for {session_id}: {e}")
        else:
            print(f"Note: No swings found for {session_id} (or processing failed).")

    if all_swings:
        MASTER_DF = pd.concat(all_swings, ignore_index=True)
        print(f"\nSUCCESS: Loaded {len(MASTER_DF)} total swings from {len(all_swings)} sessions.")
        
        # Save Master Dataset
        save_path = os.path.join(OUT_DIR, 'MASTER_TRAINING_DATA.csv')
        MASTER_DF.to_csv(save_path, index=False)
        print(f"Saved Master Training Data to: {save_path}")
        
        # --- Visualization: The "Efficiency" Proof ---
        # Plotting Cost (EMG RMS) vs. Time to see if it drifts up
        plt.figure(figsize=(12, 6))
        
        # Calculate a crude "Fatigue Cost" = RMS / Gyro_Peak
        # (Using safe division)
        MASTER_DF['cost_index'] = MASTER_DF['rms'] / (MASTER_DF['imu_peak_mag'].replace(0, 1))
        
        plt.scatter(MASTER_DF['t_start_ms'] / 60000.0, MASTER_DF['cost_index'], 
                    alpha=0.5, c=MASTER_DF['mdf'], cmap='viridis')
        
        plt.colorbar(label='Median Frequency (Hz) - Lower is Fatigue')
        plt.xlabel('Time (minutes)')
        plt.ylabel('Cost Index (RMS / SwingSpeed)')
        plt.title('Neuromechanical Efficiency over Time (All Sessions)')
        plt.grid(True, alpha=0.3)
        plt.show()
        
    else:
        print("No swing features found across sessions.")
else:
    print("Please run Cell 4 first.")